In [10]:
# === Imports & Config (omni moderation) ===
import os
import csv
import glob
import json
import time
import random
from pathlib import Path
from typing import Dict, Any, Iterable, Tuple, List, Set

from dotenv import load_dotenv
from tqdm import tqdm
from openai import OpenAI

# Input/output
INPUT_GLOB = "clean_enriched_*.jsonl"   # pattern for your clean datasets
OUTPUT_DIR = Path(".")                  # where scored CSVs & state live

# Moderation model and text truncation
MODEL = "omni-moderation-latest"        # omni moderation model
MAX_TXT_CHARS = 2000                    # chars per text sent to the model

# Batching (tuned conservatively for omni)
BATCH_SIZE = 16  # fixed batch size tuned for omni
SLEEP_BETWEEN_CALLS = 5.0  # seconds between successful moderation calls                         # can bump to 48/64 if stable
STATE_BASENAME = "scored_{sub}.__keys.json"

# Logging / progress
VERBOSE = False                         # if True, prints retry/backoff details
LOG_EVERY = None                        # or e.g. 5000 for occasional summary

# Progress display
SHOW_ROW_BAR = False                    # if True, shows 'Rows processed' tqdm bar; otherwise only 'Items scored'

# Logging / progress

# Record types to score
TYPES_TO_SCORE: Tuple[str, ...] = ("post", "comment")

# OpenAI client
load_dotenv(override=True)
client = OpenAI()  # uses OPENAI_API_KEY from env

print("OpenAI client ready. Key present:", bool(os.getenv("OPENAI_API_KEY")))


OpenAI client ready. Key present: True


In [11]:
# === Helpers: keys, text extraction, state, flattening ===

def rec_key(r: Dict[str, Any]) -> str:
    """Stable unique key per post/comment."""
    if r.get("record_type") == "comment":
        return f"t1_{r.get('comment_id')}"
    return f"t3_{r.get('post_id')}"


def text_of(r: Dict[str, Any]) -> str:
    """
    Build the text to send to moderation:
      - Posts: body or title
      - Comments: body
    Truncated to MAX_TXT_CHARS.
    """
    if r.get("record_type") == "post":
        txt = (r.get("body") or r.get("title") or "") or ""
    else:
        txt = r.get("body") or ""
    txt = str(txt).strip()
    if len(txt) > MAX_TXT_CHARS:
        txt = txt[:MAX_TXT_CHARS]
    return txt


def load_state(path: Path) -> Set[str]:
    """
    Load already-scored keys from JSON.
    Returns empty set if file missing or invalid.
    """
    if not path.exists():
        return set()
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return set(data)
        if isinstance(data, dict) and "keys" in data:
            return set(data["keys"])
        return set()
    except Exception as e:
        tqdm.write(f"[load_state] Failed to read {path.name}: {e}. Treating as empty.")
        return set()


def save_state(path: Path, done: Set[str]) -> None:
    """Save already-scored keys to JSON (atomic)."""
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    try:
        with tmp_path.open("w", encoding="utf-8") as f:
            json.dump(sorted(done), f)
        tmp_path.replace(path)
    except Exception as e:
        tqdm.write(f"[save_state] Failed to write {path.name}: {e}")


def flatten_record(rec: Dict[str, Any]) -> Dict[str, Any]:
    """
    Flatten a record + moderation dict into one flat row for CSV.
    - Copies scalar top-level fields.
    - Expands moderation categories & scores into columns.
    """
    flat: Dict[str, Any] = {}

    # scalar fields
    for k, v in rec.items():
        if k == "moderation":
            continue
        if isinstance(v, (str, int, float, bool)) or v is None:
            flat[k] = v

    mod = rec.get("moderation")
    if isinstance(mod, dict):
        flat["moderation_flagged"] = bool(mod.get("flagged", False))

        cats = mod.get("categories") or {}
        for ck, cv in cats.items():
            flat[f"moderation_cat_{ck}"] = bool(cv)

        scores = mod.get("category_scores") or {}
        for sk, sv in scores.items():
            flat[f"moderation_score_{sk}"] = sv
    else:
        if rec.get("moderation_skipped"):
            flat["moderation_skipped"] = True
        if rec.get("moderation_error"):
            flat["moderation_error"] = rec.get("moderation_error")

    return flat

In [12]:
# === Moderation helper with retry/backoff (omni) ===

def moderate_batch_with_retry(texts: List[str]) -> List[Dict[str, Any]]:
    """
    Call omni-moderation-latest with retries & backoff.
    Returns list of dicts: {"flagged", "categories", "category_scores"} per input.

    - Retries on rate limit / timeout up to 5 times.
    - If it still fails, raises → caller *skips writing* and leaves those
      items unscored so they can be retried next run.
    """
    attempt = 0

    while True:
        try:
            resp = client.moderations.create(
                model=MODEL,
                input=texts,
            )

            out: List[Dict[str, Any]] = []
            for r in resp.results:
                out.append({
                    "flagged": bool(r.flagged),
                    "categories": dict(r.categories),
                    "category_scores": dict(r.category_scores),
                })
            return out

        except Exception as e:
            msg = str(e)
            is_rate = ("429" in msg) or ("Too Many Requests" in msg) or ("rate limit" in msg.lower())
            is_timeout = ("timeout" in msg.lower())
            transient = is_rate or is_timeout

            attempt += 1

            if transient and attempt <= 5:
                base = 2 ** attempt
                wait = min(60.0, base + random.uniform(0, 2))
                if VERBOSE:
                    tqdm.write(
                        f"[moderate_batch_with_retry] transient error "
                        f"({msg[:60]}...), attempt {attempt}/5, sleep {wait:.1f}s"
                    )
                time.sleep(wait)
                continue

            tqdm.write(f"[moderate_batch_with_retry] PERMANENT error: {msg[:200]}")
            raise

In [13]:
# === Optional: summary of clean_enriched_*.jsonl files ===

def summarize_clean_datasets() -> Dict[str, Dict[str, int]]:
    """Print simple summary of each clean_enriched_*.jsonl."""
    inputs = sorted(glob.glob(INPUT_GLOB))
    summary: Dict[str, Dict[str, int]] = {}

    if not inputs:
        print("No clean_enriched_*.jsonl files found.")
        return summary

    print(f"Found {len(inputs)} subreddit dataset(s).\n")

    print("subreddit\traw_posts\traw_comments\traw_total\tclean_total\tremoved\tpct_removed\tclean_files\tpost_files\tcomment_files")
    for path in inputs:
        sub = Path(path).stem.replace("clean_enriched_", "")
        post_ct = 0
        comment_ct = 0
        total_lines = 0
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                total_lines += 1
                rec = json.loads(line)
                if rec.get("record_type") == "post":
                    post_ct += 1
                elif rec.get("record_type") == "comment":
                    comment_ct += 1

        summary[sub] = {
            "posts": post_ct,
            "comments": comment_ct,
            "total": post_ct + comment_ct,
            "total_lines": total_lines,
        }

        print(f"{sub}\tNaN\tNone\tNaN\t{total_lines:.1f}\tNaN\tNaN\tclean_enriched_{sub}.jsonl\t\t")

    print("\n=== Subreddit Dataset Summary ===")
    for sub, info in summary.items():
        print(
            f"{sub}: posts={info['posts']:,}  comments={info['comments']:,}  "
            f"total={info['total']:,}  (lines={info['total_lines']:,})"
        )

    return summary

In [14]:
# simple dummy progress bar when we don't want to show rows
class _DummyBar:
    def update(self, n):
        pass
    def close(self):
        pass

# === Scoring function with tqdm progress bars & resume via state (omni) ===

def score_file_to_csv_with_progress(
    in_path: Path,
    out_csv: Path,
    state_path: Path,
    types_to_score: Tuple[str, ...] = TYPES_TO_SCORE,
    total_lines: int | None = None,
) -> Tuple[int, int, int]:
    """
    Score one clean_enriched_<sub>.jsonl file and append to scored_<sub>.csv.

    - Respects state in scored_<sub>.__keys.json (stop/start safe).
    - Uses fixed BATCH_SIZE tuned for omni-moderation.
    - Retries on rate limit/timeout in moderate_batch_with_retry.
    - Does NOT write error-only rows.
    Returns:
        (seen_for_scoring, newly_scored, rows_written)
    """
    batch_size = int(BATCH_SIZE)

    # --- load state ---
    done = load_state(state_path)
    n_seen = 0
    n_scored = 0
    n_rows = 0

    # --- how many actually need scoring? ---
    to_score_total = 0
    with in_path.open("r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            rt = rec.get("record_type")
            if rt not in types_to_score:
                continue
            key = rec_key(rec)
            txt = text_of(rec)
            if txt and (key not in done):
                to_score_total += 1

    if to_score_total == 0:
        tqdm.write(f"[{in_path.name}] nothing to score (all items already in state).")
        return 0, 0, 0

    # --- estimate total lines for row bar ---
    if total_lines is None:
        with in_path.open("r", encoding="utf-8") as f:
            total_lines = sum(1 for _ in f)

    # Progress bars: optionally hide the row bar so only 'Items scored' is visible
    if SHOW_ROW_BAR:
        p_rows = tqdm(total=total_lines, desc=f"Rows processed ({in_path.name})", unit="row")
    else:
        p_rows = _DummyBar()

    p_scored = tqdm(total=to_score_total, desc=f"Items scored ({in_path.name})", unit="item")

    # --- CSV writer ---
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    header_written = out_csv.exists()
    f_out = out_csv.open("a", newline="", encoding="utf-8")
    writer = None

    # --- batch buffers ---
    batch_recs: List[Dict[str, Any]] = []
    batch_keys: List[str] = []
    batch_texts: List[str] = []

    def flush_batch():
        nonlocal batch_recs, batch_keys, batch_texts
        nonlocal writer, header_written, n_scored, n_rows, done

        if not batch_texts:
            return

        try:
            mods = moderate_batch_with_retry(batch_texts)
        except Exception as e:
            msg = str(e)[:200]
            tqdm.write(f"[score_file] fatal batch error, skipping {len(batch_texts)} items this run: {msg}")
            # IMPORTANT: do NOT mark as done or write anything; they'll be retried next run.
            batch_recs = []
            batch_keys = []
            batch_texts = []
            return

        if len(mods) != len(batch_recs):
            tqdm.write("[score_file] WARNING: mods length mismatch; skipping this batch.")
            batch_recs = []
            batch_keys = []
            batch_texts = []
            return

        for rec, key, mod in zip(batch_recs, batch_keys, mods):
            rec["moderation"] = {
                "flagged": bool(mod.get("flagged", False)),
                "categories": mod.get("categories", {}),
                "category_scores": mod.get("category_scores", {}),
            }

            row = flatten_record(rec)

            if writer is None:
                writer = csv.DictWriter(f_out, fieldnames=list(row.keys()))
                if not header_written:
                    writer.writeheader()
                    header_written = True

            writer.writerow(row)
            n_rows += 1
            n_scored += 1
            done.add(key)

        p_scored.update(len(batch_recs))
        save_state(state_path, done)

        if LOG_EVERY and (n_scored % LOG_EVERY == 0):
            tqdm.write(f"[progress] {in_path.name}: scored={n_scored:,} rows_written={n_rows:,}")


        batch_recs = []
        batch_keys = []
        batch_texts = []

        # fixed sleep between successful batches to respect rate limits
        if SLEEP_BETWEEN_CALLS and SLEEP_BETWEEN_CALLS > 0:
            time.sleep(SLEEP_BETWEEN_CALLS)

    # --- main loop ---
    with in_path.open("r", encoding="utf-8") as f:
        for line in f:
            p_rows.update(1)

            rec = json.loads(line)
            rt = rec.get("record_type")

            if rt not in types_to_score:
                continue

            key = rec_key(rec)
            if key in done:
                continue  # already scored in previous runs

            txt = text_of(rec)
            if not txt:
                continue

            n_seen += 1
            batch_recs.append(rec)
            batch_keys.append(key)
            batch_texts.append(txt)

            if len(batch_texts) >= batch_size:
                flush_batch()

    if batch_texts:
        flush_batch()

    p_rows.close()
    p_scored.close()
    f_out.close()
    save_state(state_path, done)

    return n_seen, n_scored, n_rows


In [15]:
# === Sampling helper: cap each subreddit at max_rows (e.g., 100,000) ===
import math

def get_sampled_path_if_needed(in_path: Path, max_rows: int = 100_000) -> Path:
    """
    If in_path has more than max_rows records, create (or reuse) a sampled file
    that keeps approximately the same post/comment ratio but with at most max_rows lines.

    - Uses the naming pattern: clean_enriched_<subreddit>_sampled100k.jsonl
    - If such a file already exists, it is reused (no re-sampling),
      so previously scored progress is not invalidated.
    - Subreddits with <= max_rows lines keep their original file unchanged.
    """
    # If already a sampled file, just use it
    if in_path.stem.endswith("_sampled100k"):
        return in_path

    # Count total lines quickly
    total_lines = 0
    with in_path.open("r", encoding="utf-8") as f:
        for _ in f:
            total_lines += 1

    if total_lines <= max_rows:
        print(f"[sample] {in_path.name}: total={total_lines:,} <= {max_rows:,} → no sampling, use original file.")
        return in_path

    sampled_path = in_path.with_name(in_path.stem + "_sampled100k.jsonl")
    if sampled_path.exists():
        print(f"[sample] {in_path.name}: using existing {sampled_path.name}.")
        return sampled_path

    print(f"[sample] {in_path.name}: total={total_lines:,} > {max_rows:,} → creating sampled file {sampled_path.name}...")

    # First pass: collect indices for posts vs comments
    post_idxs = []
    comment_idxs = []
    with in_path.open("r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            try:
                rec = json.loads(line)
            except Exception:
                continue
            rt = rec.get("record_type")
            if rt == "post":
                post_idxs.append(idx)
            elif rt == "comment":
                comment_idxs.append(idx)

    n_posts = len(post_idxs)
    n_comments = len(comment_idxs)
    total_records = n_posts + n_comments

    if total_records == 0:
        # Fallback: no clear record_type info; just take first max_rows lines
        print(f"[sample] {in_path.name}: no post/comment breakdown → copying first {max_rows:,} lines.")
        with in_path.open("r", encoding="utf-8") as fin, sampled_path.open("w", encoding="utf-8") as fout:
            for i, line in enumerate(fin):
                if i >= max_rows:
                    break
                fout.write(line)
        return sampled_path

    # Desired counts based on ratio
    posts_target = round(max_rows * n_posts / total_records)
    comments_target = max_rows - posts_target

    random.seed(42)  # stable sampling across runs
    posts_keep = set(random.sample(post_idxs, min(posts_target, n_posts)))
    comments_keep = set(random.sample(comment_idxs, min(comments_target, n_comments)))
    keep_idxs = posts_keep | comments_keep

    # Second pass: write only selected indices
    with in_path.open("r", encoding="utf-8") as fin, sampled_path.open("w", encoding="utf-8") as fout:
        for idx, line in enumerate(fin):
            if idx in keep_idxs:
                fout.write(line)

    kept = sum(1 for _ in sampled_path.open("r", encoding="utf-8"))
    print(f"[sample] {in_path.name}: wrote {kept:,} lines to {sampled_path.name}.")
    return sampled_path


In [16]:
# === Prefill helper: reuse rows from full scored CSV into sampled scored CSV ===

def prefill_sample_from_full(base_sub: str, sample_jsonl: Path, out_csv: Path, state_path: Path):
    """
    If scored_<base_sub>.csv exists (full dataset), copy any rows whose keys appear
    in the sampled JSONL into the sampled scored CSV, and mark their keys as done
    in the sampled state file so they are not re-scored.

    This lets us reuse HollowKnight/WoW/r_gaming scores from the big runs.
    """
    full_csv = OUTPUT_DIR / f"scored_{base_sub}.csv"
    if not full_csv.exists():
        return

    # Collect all keys present in the sampled JSONL
    sample_keys = set()
    with sample_jsonl.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except Exception:
                continue
            rt = rec.get("record_type")
            if rt not in TYPES_TO_SCORE:
                continue
            key = rec_key(rec)
            sample_keys.add(key)

    # Load existing sampled state (keys already done)
    done = load_state(state_path)

    # Also load keys already in sampled CSV (if it exists)
    if out_csv.exists():
        with out_csv.open("r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                try:
                    k = rec_key(row)
                except Exception:
                    continue
                done.add(k)

    # Now copy matching rows from full CSV into sampled CSV
    if not full_csv.exists():
        return

    with full_csv.open("r", encoding="utf-8", newline="") as f_full,          out_csv.open("a", encoding="utf-8", newline="") as f_sample:

        reader_full = csv.DictReader(f_full)
        writer_sample = None

        # If sampled CSV is empty, we should write header when we first write
        sample_empty = (os.path.getsize(out_csv) == 0) if out_csv.exists() else True

        for row in reader_full:
            try:
                k = rec_key(row)
            except Exception:
                continue
            if k in sample_keys and k not in done:
                if writer_sample is None:
                    writer_sample = csv.DictWriter(f_sample, fieldnames=reader_full.fieldnames)
                    if sample_empty:
                        writer_sample.writeheader()
                        sample_empty = False
                writer_sample.writerow(row)
                done.add(k)

    save_state(state_path, done)
    print(f"[prefill] {base_sub}: prefilled {len(done)} keys into sampled state (including any previously existing).")


In [ ]:
# === Run scoring for all clean_enriched_*.jsonl files (using sampled where available) ===

summary = summarize_clean_datasets()

inputs = sorted(glob.glob(INPUT_GLOB))
print(f"Found {len(inputs)} subreddit dataset(s).\n")

# Group paths by logical base subreddit (strip optional '_sampled100k')
grouped = {}
for path in inputs:
    p = Path(path)
    stem = p.stem.replace("clean_enriched_", "")
    if stem.endswith("_sampled100k"):
        base = stem[:-len("_sampled100k")]
        grouped.setdefault(base, {})["sampled"] = p
    else:
        base = stem
        grouped.setdefault(base, {})["full"] = p

print("=== Subreddit Dataset Summary ===")
for base, choice in grouped.items():
    info = summary.get(base, {})
    posts = info.get("posts")
    comments = info.get("comments")
    total = info.get("total_lines")
    print(f"{base}: posts={posts:,}  comments={comments:,}  total={total:,}")

total_counts = {"seen": 0, "scored": 0, "rows": 0}

for base_sub, choice in grouped.items():
    # Decide which input file to use
    if "sampled" in choice:
        in_path = choice["sampled"]
        use_sampled = True
    else:
        # possibly create a sampled file if needed
        in_path = get_sampled_path_if_needed(choice["full"], max_rows=100_000)
        use_sampled = in_path.stem.endswith("_sampled100k")

    # Decide output CSV and state filenames
    if use_sampled:
        out_csv = OUTPUT_DIR / f"scored_{base_sub}_sampled100k.csv"
        state_path = OUTPUT_DIR / f"scored_{base_sub}_sampled100k.__keys.json"
    else:
        out_csv = OUTPUT_DIR / f"scored_{base_sub}.csv"
        state_path = OUTPUT_DIR / STATE_BASENAME.format(sub=base_sub)

    print(f"\n=== {base_sub} ({'sampled100k' if use_sampled else 'full dataset'}) ===")
    print("Input:", in_path.name)
    print("Out  :", out_csv.name)
    print("State:", state_path.name)

    # If this is a sampled run and we have a full scored CSV, prefill from it
    if use_sampled:
        prefill_sample_from_full(base_sub, in_path, out_csv, state_path)

    # Let scoring function handle state-based skipping
    n_seen, n_scored, n_rows = score_file_to_csv_with_progress(
        in_path=in_path,
        out_csv=out_csv,
        state_path=state_path,
        types_to_score=TYPES_TO_SCORE,
        total_lines=None,
    )

    print(f"[{base_sub}] seen_for_scoring={n_seen:,}  newly_scored={n_scored:,}  rows_written={n_rows:,}")
    total_counts["seen"] += n_seen
    total_counts["scored"] += n_scored
    total_counts["rows"] += n_rows

print("\nTOTAL:", total_counts)


Found 12 subreddit dataset(s).

subreddit	raw_posts	raw_comments	raw_total	clean_total	removed	pct_removed	clean_files	post_files	comment_files
AlbionOnline	NaN	None	NaN	25277.0	NaN	NaN	clean_enriched_AlbionOnline.jsonl		
DarkSouls	NaN	None	NaN	30981.0	NaN	NaN	clean_enriched_DarkSouls.jsonl		
ElderScrollsOnline	NaN	None	NaN	55344.0	NaN	NaN	clean_enriched_ElderScrollsOnline.jsonl		
HollowKnight	NaN	None	NaN	488914.0	NaN	NaN	clean_enriched_HollowKnight.jsonl		
HollowKnight_sampled100k	NaN	None	NaN	100000.0	NaN	NaN	clean_enriched_HollowKnight_sampled100k.jsonl		
LiesofP	NaN	None	NaN	51082.0	NaN	NaN	clean_enriched_LiesofP.jsonl		
Ninesols	NaN	None	NaN	8798.0	NaN	NaN	clean_enriched_Ninesols.jsonl		
Palia	NaN	None	NaN	80158.0	NaN	NaN	clean_enriched_Palia.jsonl		
WoW	NaN	None	NaN	387803.0	NaN	NaN	clean_enriched_WoW.jsonl		
WoW_sampled100k	NaN	None	NaN	100000.0	NaN	NaN	clean_enriched_WoW_sampled100k.jsonl		
r_gaming	NaN	None	NaN	518355.0	NaN	NaN	clean_enriched_r_gaming.jsonl		


                                                                                                       
Items scored (clean_enriched_LiesofP.jsonl):  90%|████████▉ | 45808/51082 [11:31:13<48:29,  1.81item/s]          

r_gaming_sampled100k	NaN	None	NaN	100000.0	NaN	NaN	clean_enriched_r_gaming_sampled100k.jsonl		

=== Subreddit Dataset Summary ===
AlbionOnline: posts=2,191  comments=23,086  total=25,277  (lines=25,277)
DarkSouls: posts=2,272  comments=28,709  total=30,981  (lines=30,981)
ElderScrollsOnline: posts=2,711  comments=52,633  total=55,344  (lines=55,344)
HollowKnight: posts=43,418  comments=445,496  total=488,914  (lines=488,914)
HollowKnight_sampled100k: posts=8,880  comments=91,120  total=100,000  (lines=100,000)
LiesofP: posts=2,869  comments=48,213  total=51,082  (lines=51,082)
Ninesols: posts=648  comments=8,150  total=8,798  (lines=8,798)
Palia: posts=5,022  comments=75,136  total=80,158  (lines=80,158)
WoW: posts=12,442  comments=375,361  total=387,803  (lines=387,803)
WoW_sampled100k: posts=3,208  comments=96,792  total=100,000  (lines=100,000)
r_gaming: posts=0  comments=518,355  total=518,355  (lines=518,355)
r_gaming_sampled100k: posts=0  comments=100,000  total=100,000  (lines=1

                                                                                                       
Items scored (clean_enriched_LiesofP.jsonl):  90%|████████▉ | 45808/51082 [11:31:14<48:29,  1.81item/s]          

[sample] clean_enriched_DarkSouls.jsonl: total=30,981 <= 100,000 → no sampling, use original file.

=== DarkSouls (full dataset) ===
Input: clean_enriched_DarkSouls.jsonl
Out  : scored_DarkSouls.csv
State: scored_DarkSouls.__keys.json
[clean_enriched_DarkSouls.jsonl] nothing to score (all items already in state).
[DarkSouls] seen_for_scoring=0  newly_scored=0  rows_written=0
[sample] clean_enriched_ElderScrollsOnline.jsonl: total=55,344 <= 100,000 → no sampling, use original file.

=== ElderScrollsOnline (full dataset) ===
Input: clean_enriched_ElderScrollsOnline.jsonl
Out  : scored_ElderScrollsOnline.csv
State: scored_ElderScrollsOnline.__keys.json


...

[clean_enriched_ElderScrollsOnline.jsonl] nothing to score (all items already in state).
[ElderScrollsOnline] seen_for_scoring=0  newly_scored=0  rows_written=0

=== HollowKnight (sampled100k) ===
Input: clean_enriched_HollowKnight_sampled100k.jsonl
Out  : scored_HollowKnight_sampled100k.csv
State: scored_HollowKnight_sampled100k.__keys.json
[prefill] HollowKnight: prefilled 99712 keys into sampled state (including any previously existing).











                                                                                                       
                                                                                                              

Items scored (clean_enriched_LiesofP.jsonl):  90%|████████▉ | 45808/51082 [11:34:09<48:29,  1.81item/s]          
                                                                                                       
                                                                                                              

Items scored (clean_enriched_LiesofP.jsonl):  90%|████████▉ | 45808/51082 [11:34:09<48:29,  1.81item/s]          


[moderate_batch_with_retry] PERMANENT error: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'invalid_request_error', 'param': None, 'code': None}}
[score_file] fatal batch error, skipping 16 items this run: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'invalid_request_error', 'param': None, 'code': None}}











Items scored (clean_enriched_HollowKnight_sampled100k.jsonl):  94%|█████████▍| 272/288 [05:13<00:18,  1.15s/item]


[HollowKnight] seen_for_scoring=288  newly_scored=272  rows_written=272
[sample] clean_enriched_LiesofP.jsonl: total=51,082 <= 100,000 → no sampling, use original file.

=== LiesofP (full dataset) ===
Input: clean_enriched_LiesofP.jsonl
Out  : scored_LiesofP.csv
State: scored_LiesofP.__keys.json
